In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os 

In [ ]:
my_path = os.path.join("data", "customer_data.csv")
customer_df  = pd.read_csv(my_path, sep="\t")

In [ ]:
customer_df.shape

In [ ]:
pd.set_option('display.max_columns', None)
customer_df.head()

In [ ]:
customer_df.info()

In [ ]:
customer_df.describe(include='all')

In [ ]:
columns_str_dtype = customer_df.columns[customer_df.dtypes == 'str']
columns_str_dtype

In [ ]:
for col in columns_str_dtype:
  unique_values = sorted(customer_df[col].unique())
  print(f"{col}: length {len(unique_values)}")
  print(unique_values, '\n')

In [ ]:
customer_df.isna().sum()

In [ ]:
customer_df = customer_df.dropna()
customer_df.isna().sum()

In [ ]:
customer_df["birth_year"] = 2023 - customer_df["birth_year"]

In [ ]:
customer_df = customer_df.rename(columns={"birth_year": "age"})
customer_df.head()

In [ ]:
data_amount_total = (
    customer_df['amount_alcohol']
    + customer_df['amount_fruit']
    + customer_df['amount_meat']
    + customer_df['amount_fish']
    + customer_df['amount_snack']
    + customer_df['amount_general']
)


In [ ]:
index_amount_general = customer_df.columns.get_loc("amount_general")

In [ ]:
customer_df.insert(
  loc = index_amount_general + 1,
  column = "amount_total",
  value = data_amount_total
)
customer_df.head()

In [ ]:
num_purchase_total = (
    customer_df['num_purchase_web']
    + customer_df['num_purchase_store']
    + customer_df['num_purchase_discount']
)

index_num_purchase_discount = customer_df.columns.get_loc('num_purchase_discount')
customer_df.insert(
    loc=index_num_purchase_discount + 1,
    column='num_purchase_total',
    value=num_purchase_total,
)
customer_df.head()


In [ ]:
customer_df["revenue"].describe()

In [ ]:
customer_df = customer_df.drop(columns=["ID", "revenue"])
customer_df.head()

In [ ]:
# ---------------------Data exploration and preprocessing complete---------------------

In [ ]:
plt.rc('font', family='AppleGothic')

In [ ]:
plt.rcParams["figure.figsize"] = (10, 5)
sns.histplot(data = customer_df['age'])
plt.title("Age Distribution of Customers")
plt.xlabel("Age")
plt.ylabel("Customer Count")

In [ ]:
customer_df.sort_values(by="age", ascending=False)

In [ ]:
customer_df = customer_df[customer_df["age"] < 100]

In [ ]:
sns.histplot(data=customer_df["age"])
plt.title("Age Distribution of Customers")
plt.xlabel("Age")
plt.ylabel("Customer Count")

In [ ]:
age_bins = list(range(10,81,10))
age_bins

In [ ]:
age_labels = [f'{x}s' for x in age_bins[:-1]]
age_labels

In [ ]:
age_group = pd.cut(customer_df["age"], bins=age_bins, labels=age_labels, right=False)

In [ ]:
customer_df.insert(
  loc = customer_df.columns.get_loc("age") + 1,
  column = "age_group",
  value = age_group
)
customer_df.head()

In [ ]:
customer_df["age_group"].value_counts()

In [ ]:
age_group_replace_dict = {
    '10s': 'under_20s',
    '20s': 'under_20s',
    '60s': 'over_60',
    '70s': 'over_60',
}

customer_df["age_group"] = customer_df["age_group"].astype(str).replace(age_group_replace_dict)


In [ ]:
customer_df['age_group'].value_counts()

In [ ]:
sns.histplot(data=customer_df["annual_income"])
plt.title("Annual Income Distribution of Customers")
plt.xlabel("Annual Income")
plt.ylabel("Customer Count")

In [ ]:
sns.boxplot(data=customer_df, x="annual_income")

In [ ]:
income = customer_df["annual_income"]
q1 = income.quantile(0.25)
q3 = income.quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

normal_condition = (lower_bound <= income) & (income <= upper_bound)
customer_df = customer_df[normal_condition]

In [ ]:
sns.boxplot(data=customer_df, x='annual_income')

In [ ]:
#-----------Demographic analysis complete----------------

In [ ]:
num_grades = 3

In [ ]:
grade_labels = list(range(1,num_grades + 1))
grade_labels

In [ ]:
recency_grade = pd.qcut(x=customer_df["recency"], q=num_grades, labels=grade_labels[::-1])
customer_df["recency_grade"] = recency_grade


In [ ]:
customer_df['recency_grade'].value_counts()

In [ ]:
groupby_recency_grade = customer_df.groupby("recency_grade").sum(numeric_only=True).reset_index()
groupby_recency_grade

In [ ]:
groupby_recency_grade["amount_total"].plot(kind="pie")

In [ ]:
groupby_recency_grade["amount_total"].plot(
  kind="pie",
  autopct='%.1f%%',
  labels = [f"{x} level" for x in grade_labels[::-1]],
  title = "Total Amount by Recency Grade",
  ylabel = ""
)

In [ ]:
customer_df["frequency_grade"] = pd.qcut(
  x=customer_df["num_purchase_total"],
  q=num_grades,
  labels=grade_labels
)

In [ ]:
customer_df["frequency_grade"].value_counts()

In [ ]:
groupby_frequency_grade = customer_df.groupby("frequency_grade").sum(numeric_only=True).reset_index()
groupby_frequency_grade["amount_total"].plot(
  kind="pie",
  autopct='%.1f%%',
  labels = [f"{x} level" for x in grade_labels],
  title = "Total Amount by Frequency Grade",
  ylabel = ""
)

In [ ]:
customer_df["monetary_grade"] = pd.qcut(
  x=customer_df["amount_total"],
  q=num_grades,
  labels=grade_labels
)

In [ ]:
customer_df["monetary_grade"].value_counts()


In [ ]:
groupby_monetary_grade = customer_df.groupby("monetary_grade").sum(numeric_only=True).reset_index()
groupby_monetary_grade["amount_total"].plot(
  kind="pie",
  autopct='%.1f%%',
  labels = [f"{x} level" for x in grade_labels],
  title = "Total Amount by Monetary Grade",
  ylabel = ""
)

In [ ]:
weight = {}
weight["recency"] = 1/3
weight["frequency"] = 1/3
weight["monetary"] = 1/3


In [ ]:
customer_df["rfm_score"] = (
  weight["recency"] * customer_df["recency_grade"].astype(int)
  + weight["frequency"] * customer_df["frequency_grade"].astype(int)
  + weight["monetary"] * customer_df["monetary_grade"].astype(int)
)

In [ ]:
def rfm_segment_bins(x):
  if x < 5/3:
    return 1
  elif x <= 7/3:
    return 2
  else:
    return 3
customer_df["rfm_segment"] = customer_df["rfm_score"].apply(rfm_segment_bins)

In [ ]:
customer_df["rfm_segment"].value_counts()

In [ ]:
groupby_rfm_segment = customer_df.groupby("rfm_segment").sum(numeric_only=True).reset_index()
groupby_rfm_segment["amount_total"].plot(
  kind="pie",
  autopct='%.1f%%',
  labels = [f"{x} level" for x in grade_labels],
  title = "Total Amount by RFM Segment",
  ylabel = ""
)

In [ ]:
#reset weight
weight["recency"] = 0.2
weight["frequency"] = 0.4
weight["monetary"] = 0.4

customer_df["rfm_score"] = (
  weight["recency"] * customer_df["recency_grade"].astype(int)
  + weight["frequency"] * customer_df["frequency_grade"].astype(int)
  + weight["monetary"] * customer_df["monetary_grade"].astype(int)
)

customer_df["rfm_segment"] = customer_df["rfm_score"].apply(rfm_segment_bins)

In [ ]:
customer_df['rfm_segment'].value_counts()


In [ ]:
groupby_rfm_segment = customer_df.groupby("rfm_segment").sum(numeric_only=True).reset_index()
groupby_rfm_segment["amount_total"].plot(
  kind="pie",
  autopct='%.1f%%',
  labels = [f"{x} level" for x in grade_labels],
  title = "Total Amount by RFM Segment",
  ylabel = ""
) 

In [ ]:
#---------------------------RFM customer segmentation complete------------------

In [ ]:
groupby_rfm_segment_age_group = customer_df.groupby(['rfm_segment', 'age_group']).size().reset_index()
groupby_rfm_segment_age_group

In [ ]:
groupby_rfm_segment_age_group = groupby_rfm_segment_age_group.rename(columns={0: 'num_customers'})

In [ ]:
for i_segment in range(1, num_grades + 1):
    age_group_dist = groupby_rfm_segment_age_group[
        groupby_rfm_segment_age_group['rfm_segment'] == i_segment
    ]
    age_group_dist['num_customers'].plot(
        kind='pie',
        autopct='%.1f%%',
        labels=age_group_dist['age_group'].unique(),
        title=f'{i_segment} level RFM Segment Age Group Distribution',
        ylabel='',
    )
    plt.show()



In [ ]:
groupby_rfm_segment_marital = customer_df.groupby(['rfm_segment', 'marital_status']).size().reset_index()
groupby_rfm_segment_marital = groupby_rfm_segment_marital.rename(columns={0: 'num_customers'})

for i_segment in range(1, num_grades + 1):
    marital_status_dist = groupby_rfm_segment_marital[
                groupby_rfm_segment_marital['rfm_segment'] == i_segment
        ]
    marital_status_dist['num_customers'].plot(
        kind='pie',
        autopct='%.1f%%',
        labels=marital_status_dist['marital_status'].unique(),
        title=f'{i_segment} level RFM Segment Marital Status Distribution',
        ylabel='',
    )
    plt.show()

In [ ]:
groupby_rfm_segment_children = customer_df.groupby(['rfm_segment', 'children']).size().reset_index()
groupby_rfm_segment_children = groupby_rfm_segment_children.rename(columns={0: 'num_customers'})

for i_segment in range(1, num_grades + 1):
    children_dist = groupby_rfm_segment_children[
                groupby_rfm_segment_children['rfm_segment'] == i_segment
        ]
    children_dist['num_customers'].plot(
        kind='pie',
        autopct='%.1f%%',
        labels=[f'{i}명' for i in children_dist['children'].unique()],
        title=f'{i_segment} level RFM Segment Number of Dependent Children Distribution',
        ylabel='',
    )
    plt.show()

In [ ]:
groupby_rfm_segment = customer_df.groupby('rfm_segment').sum(numeric_only=True).reset_index()

In [ ]:
selected_columns = [
    col
    for col in groupby_rfm_segment.columns
    if col.startswith('amount_') and col != 'amount_total'
]
selected_columns.append('rfm_segment')
selected_columns

In [ ]:
amount_sum_per_product = groupby_rfm_segment[selected_columns]
amount_sum_per_product = amount_sum_per_product.set_index('rfm_segment')
amount_sum_per_product

In [ ]:
for i_segment in range(1, num_grades + 1):
    amount_sum_per_product.loc[i_segment].plot(
        kind='pie',
        autopct='%.1f%%',
        labels=['alcohol', 'fruit', 'meat', 'fish', 'snack', 'general'],
        title=f'{i_segment} Level RFM Segment Product Category Sales Contribution',
        ylabel='',
    )
    plt.show()

In [ ]:
groupby_rfm_segment = customer_df.groupby(['rfm_segment']).mean(numeric_only=True).reset_index()

In [ ]:
selected_columns = [f'promotion_{i}' for i in range(1, 7)]
selected_columns.append('rfm_segment')
selected_columns

In [ ]:
avg_promotion = groupby_rfm_segment[selected_columns]
avg_promotion = avg_promotion.set_index('rfm_segment')
avg_promotion

In [ ]:
avg_promotion.plot(kind='bar')
plt.title("Average Promotion Usage by RFM Segment"),
plt.xlabel("RFM Segment"),
plt.ylabel("Average Promotion Usage"),
plt.grid()


In [ ]:
#------Segment characteristics and spending behavior analysis complete----------